In [0]:
# Databricks notebook source
# COMMAND ----------
# MAGIC %run ../00_config/00_config

# COMMAND ----------
# 1. EXPLICIT PYSPARK IMPORTS
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, 
    TimestampType, DoubleType
)
from delta.tables import DeltaTable
from datetime import datetime
import uuid

# Project Salt for SHA-256 PII Hashing
SALT = "EbayMarketplace2026SaltKey!"
start_time = datetime.now()

TABLE_SILVER_LISTINGS = "ebay_pipeline.silver_listings"
TABLE_QUARANTINE = "ebay_pipeline.silver_quarantine_records"
TABLE_LOGS = "ebay_pipeline.pipeline_execution_logs"

# 2. WIDGETS FOR PARAMETERIZED BACKFILLS (Phase 2 Rubric)
dbutils.widgets.text("batch_id_filter", "ALL", "Batch ID Filter")
dbutils.widgets.text("ingestion_date_filter", "ALL", "Ingestion Date Filter YYYY-MM-DD")

BATCH_FILTER = dbutils.widgets.get("batch_id_filter").strip()
DATE_FILTER = dbutils.widgets.get("ingestion_date_filter").strip()

# COMMAND ----------
try:
    df_bronze_inc = spark.table("ebay_pipeline.bronze_ebay_listings_incremental")

    # Apply Parameterized Filtering if provided for Backfill
    if BATCH_FILTER != "ALL" and BATCH_FILTER != "":
        df_bronze_inc = df_bronze_inc.filter(F.col("batch_id") == BATCH_FILTER)
        active_param = f"batch_id={BATCH_FILTER}"
    elif DATE_FILTER != "ALL" and DATE_FILTER != "":
        df_bronze_inc = df_bronze_inc.filter(F.col("ingestion_date") == DATE_FILTER)
        active_param = f"ingestion_date={DATE_FILTER}"
    else:
        active_param = "ALL_INCREMENTAL_RECORDS"

    # 1. Type Casting & Salted SHA-256 PII Hashing on Seller Handles
    df_transformed = (
        df_bronze_inc
        .select(
            F.col("raw_payload.itemId").alias("item_id"),
            F.col("raw_payload.title").alias("title"),
            F.col("subcategory_partition").alias("subcategory"),
            F.col("raw_payload.price.value").cast(DoubleType()).alias("price"),
            F.col("raw_payload.price.currency").alias("currency"),
            F.col("raw_payload.condition").alias("condition"),
            F.to_timestamp("raw_payload.itemCreationDate").alias("item_creation_timestamp"),
            F.sha2(F.concat(F.col("raw_payload.seller"), F.lit(SALT)), 256).alias("seller_id_hashed"),
            F.col("batch_id"),
            F.current_timestamp().alias("load_timestamp")
        )
    )

    # 2. Data Quality & Quarantine Logic (Rubric: Quality/Quarantine)
    # Checks: Missing primary key, or invalid/negative price
    invalid_condition = (
        F.col("item_id").isNull() | (F.col("item_id") == "") |
        F.col("price").isNull() | (F.col("price") <= 0)
    )

    df_quarantine = (
        df_transformed
        .filter(invalid_condition)
        .withColumn(
            "rejection_reason",
            F.when(F.col("item_id").isNull() | (F.col("item_id") == ""), F.lit("Missing Primary Key item_id"))
             .when(F.col("price").isNull() | (F.col("price") <= 0), F.lit("Invalid Price (<= 0 or Null)"))
             .otherwise(F.lit("Unknown Validation Failure"))
        )
        .withColumn("quarantined_at", F.current_timestamp())
    )

    quarantine_count = df_quarantine.count()
    if quarantine_count > 0:
        (
            df_quarantine
            .write.format("delta").mode("append")
            .saveAsTable(TABLE_QUARANTINE)
        )
        print(f"Quarantined {quarantine_count} invalid records into {TABLE_QUARANTINE}")

    # 3. Deduplicate Clean Staging Records on Natural Key
    df_valid = (
        df_transformed
        .filter(~invalid_condition)
        .dropDuplicates(["item_id"])
    )

    # 4. Idempotent MERGE INTO (SCD Type 1 Upsert Pattern)
    table_exists = spark.catalog.tableExists(TABLE_SILVER_LISTINGS)

    if not table_exists:
        df_valid.write.format("delta").mode("overwrite").saveAsTable(TABLE_SILVER_LISTINGS)
        inserted = df_valid.count()
        updated = 0
    else:
        silver_delta = DeltaTable.forName(spark, TABLE_SILVER_LISTINGS)

        (
            silver_delta.alias("target")
            .merge(
                df_valid.alias("source"),
                "target.item_id = source.item_id"
            )
            .whenMatchedUpdate(set={
                "title": "source.title",
                "price": "source.price",
                "condition": "source.condition",
                "load_timestamp": "source.load_timestamp"
            })
            .whenNotMatchedInsertAll()
            .execute()
        )
        
        # Exact metrics from Delta history
        history = spark.sql(f"DESCRIBE HISTORY {TABLE_SILVER_LISTINGS} LIMIT 1").collect()
        op_metrics = history[0]["operationMetrics"]
        inserted = int(op_metrics.get("numTargetRowsInserted", 0))
        updated = int(op_metrics.get("numTargetRowsUpdated", 0))

    end_time = datetime.now()

    # 5. Log Run to Audit Table
    log_schema = StructType([
        StructField("log_id", StringType(), False),
        StructField("layer", StringType(), False),
        StructField("table_name", StringType(), False),
        StructField("parameter_processed", StringType(), True),
        StructField("execution_start_time", TimestampType(), False),
        StructField("execution_end_time", TimestampType(), True),
        StructField("status", StringType(), False),
        StructField("rows_inserted", LongType(), True),
        StructField("rows_updated", LongType(), True),
        StructField("error_message", StringType(), True)
    ])

    log_entry = [(
        str(uuid.uuid4()),
        "Bronze-to-Silver",
        TABLE_SILVER_LISTINGS,
        active_param,
        start_time,
        end_time,
        "SUCCESS",
        inserted,
        updated,
        None
    )]

    spark.createDataFrame(log_entry, schema=log_schema).write.format("delta").mode("append").saveAsTable(TABLE_LOGS)
    print(f"Silver Merge Succeeded: {inserted} inserted, {updated} updated into {TABLE_SILVER_LISTINGS} [Param: {active_param}]")

except Exception as e:
    end_time = datetime.now()
    log_entry = [(
        str(uuid.uuid4()),
        "Bronze-to-Silver",
        TABLE_SILVER_LISTINGS,
        active_param if 'active_param' in locals() else "UNKNOWN",
        start_time,
        end_time,
        "FAILURE",
        0,
        0,
        str(e)
    )]
    spark.createDataFrame(log_entry, schema=log_schema).write.format("delta").mode("append").saveAsTable(TABLE_LOGS)
    raise e


Silver Merge Succeeded: 0 inserted, 531 updated into ebay_pipeline.silver_listings [Param: ALL_INCREMENTAL_RECORDS]


In [0]:
# COMMAND ----------
# Rubric Verification: Re-run identical batch and verify 0 duplicates
count_before = spark.table("ebay_pipeline.silver_listings").count()
print(f"Silver count before re-run: {count_before}")

# Re-execute Silver logic
(
    silver_delta.alias("target")
    .merge(
        df_valid.alias("source"),
        "target.item_id = source.item_id"
    )
    .whenMatchedUpdate(set={"load_timestamp": "source.load_timestamp"})
    .whenNotMatchedInsertAll()
    .execute()
)

count_after = spark.table("ebay_pipeline.silver_listings").count()
print(f"Silver count after re-run:  {count_after}")

assert count_before == count_after, "Idempotency failed: duplicate rows created!"
print("IDEMPOTENCY PROVEN: Re-running identical load produced 0 duplicate records.")

# Inspect Execution Audit Trail
display(spark.table("ebay_pipeline.pipeline_execution_logs").orderBy("execution_start_time", ascending=False))

Silver count before re-run: 531
Silver count after re-run:  531
IDEMPOTENCY PROVEN: Re-running identical load produced 0 duplicate records.


log_id,layer,table_name,parameter_processed,execution_start_time,execution_end_time,status,rows_inserted,rows_updated,error_message
38edd230-b3f1-467f-9dd6-e6a5acfb69ed,Bronze-to-Silver,ebay_pipeline.silver_listings,ALL_INCREMENTAL_RECORDS,2026-10-16T14:00:25.805Z,2026-10-16T14:00:35.034Z,SUCCESS,0,531,null
55ba28f8-1320-43d2-962e-d6d5c95d419c,Bronze-to-Silver,ebay_pipeline.silver_listings,ALL_INCREMENTAL_RECORDS,2026-10-16T13:58:53.196Z,2026-10-16T13:59:09.530Z,SUCCESS,49,482,null
039d2514-1560-4c92-91db-b4eaec3a84d7,Raw-to-Bronze,bronze_ebay_listings_incremental,ebay_incremental_*.json,2026-10-16T13:55:01.272Z,2026-10-16T13:55:07.546Z,SUCCESS,780,0,null
c21698b5-2934-4615-abc8-b1ed44b38447,Raw-to-Bronze,bronze_ebay_listings_full,full_load/*.json,2026-10-16T13:53:49.230Z,2026-10-16T13:54:38.462Z,SUCCESS,267,0,null
e5836a7e-b3c7-4782-a29d-5ca9a2635f9c,Bronze-to-Silver,ebay_pipeline.silver_listings,ALL_INCREMENTAL_RECORDS,2026-10-11T17:47:34.974Z,2026-10-11T17:47:49.703Z,SUCCESS,129,353,null
acccbfa9-8ae5-4810-8256-585ef6e34181,Raw-to-Bronze,bronze_ebay_listings_full,full_load/*.json,2026-10-11T17:46:56.757Z,2026-10-11T17:47:03.691Z,SUCCESS,267,0,null
c1be504c-eea5-4433-ad9e-f4be907e8747,Raw-to-Bronze,bronze_ebay_listings_incremental,ebay_incremental_*.json,2026-10-11T17:46:00.245Z,2026-10-11T17:46:21.752Z,SUCCESS,728,0,null
9aa97e91-f2d5-4764-91a3-5a0f20235e5c,Bronze-to-Silver,ebay_pipeline.silver_listings,bronze_incremental_source,2026-10-10T12:42:38.453Z,2026-10-10T12:42:50.585Z,SUCCESS,73,280,null
387c9f42-6cc6-405a-8a97-90749f961e73,Raw-to-Bronze,bronze_ebay_listings_incremental,ebay_incremental_*.json,2026-10-10T12:39:52.385Z,2026-10-10T12:39:58.408Z,SUCCESS,416,0,null
a2ab6d41-d3df-489d-b589-6eeb8feb7b42,Bronze-to-Silver,ebay_pipeline.silver_listings,bronze_incremental_source,2026-10-10T12:02:09.688Z,2026-10-10T12:02:17.443Z,SUCCESS,280,0,null


In [0]:
%sql
SELECT 
    layer,
    table_name,
    parameter_processed,
    execution_start_time,
    execution_end_time,
    status,
    rows_inserted,
    rows_updated
FROM ebay_pipeline.pipeline_execution_logs
ORDER BY execution_start_time DESC;

layer,table_name,parameter_processed,execution_start_time,execution_end_time,status,rows_inserted,rows_updated
Bronze-to-Silver,ebay_pipeline.silver_listings,ALL_INCREMENTAL_RECORDS,2026-10-11T17:47:34.974Z,2026-10-11T17:47:49.703Z,SUCCESS,129,353
Raw-to-Bronze,bronze_ebay_listings_full,full_load/*.json,2026-10-11T17:46:56.757Z,2026-10-11T17:47:03.691Z,SUCCESS,267,0
Raw-to-Bronze,bronze_ebay_listings_incremental,ebay_incremental_*.json,2026-10-11T17:46:00.245Z,2026-10-11T17:46:21.752Z,SUCCESS,728,0
Bronze-to-Silver,ebay_pipeline.silver_listings,bronze_incremental_source,2026-10-10T12:42:38.453Z,2026-10-10T12:42:50.585Z,SUCCESS,73,280
Raw-to-Bronze,bronze_ebay_listings_incremental,ebay_incremental_*.json,2026-10-10T12:39:52.385Z,2026-10-10T12:39:58.408Z,SUCCESS,416,0
Bronze-to-Silver,ebay_pipeline.silver_listings,bronze_incremental_source,2026-10-10T12:02:09.688Z,2026-10-10T12:02:17.443Z,SUCCESS,280,0
Raw-to-Bronze,bronze_ebay_listings_incremental,ebay_incremental_*.json,2026-10-10T11:57:08.944Z,2026-10-10T11:57:57.031Z,SUCCESS,312,0
Raw-to-Bronze,bronze_ebay_listings_full,full_load/*.json,2026-10-10T11:20:47.505Z,2026-10-10T11:20:53.040Z,SUCCESS,267,0
Raw-to-Bronze,bronze_ebay_listings_full,full_load/*.json,2026-10-10T11:12:35.320Z,2026-10-10T11:12:40.431Z,SUCCESS,14,0


In [0]:
%sql
SELECT 
    batch_id,
    MAX(load_timestamp) AS last_silver_update,
    MAX(item_creation_timestamp) AS newest_listing_created,
    COUNT(*) AS total_listings
FROM ebay_pipeline.silver_listings
GROUP BY batch_id
ORDER BY last_silver_update DESC;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6391792249094761>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT \n    batch_id,\n    MAX(load_timestamp) AS last_silver_update,\n    MAX(item_creation_timestamp) AS newest_listing_created,\n    COUNT(*) AS total_listings\nFROM ebay_pipeline.silver_listings\nGROUP BY batch_id\nORDER BY last_silver_update DESC;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGI

In [0]:
%sql
SELECT 
    MAX(load_timestamp) AS last_silver_update,
    COUNT(*) AS total_listings
FROM ebay_pipeline.silver_listings;

last_silver_update,total_listings
2026-10-16T14:00:39.252Z,531
